# Lab Work - 11.2

**Dataset**  
`x = [10, 20, 30, 40, 50]`  
`y = [2, 4, 5, 4, 5]`  
Indices: 0, 1, 2, 3, 4

I am using a fixed random seed (`np.random.seed(0)`) so that the bootstrap samples (and therefore all downstream results) are fully reproducible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

x = np.array([10., 20., 30., 40., 50.])
y = np.array([2., 4., 5., 4., 5.])
indices = np.arange(5)

print("Original data:")
for i, (xi, yi) in enumerate(zip(x, y)):
    print(f"  index {i}: x={xi:.0f}, y={yi:.0f}")

## Q.1  Bootstrap Sampling by Hand

In [ ]:
# 02 & 03 — Draw two independent bootstrap samples (with replacement)
sample1_idx = np.random.choice(indices, size=5, replace=True)
sample2_idx = np.random.choice(indices, size=5, replace=True)

print("Bootstrap Sample 1 indices:", sample1_idx.tolist())
print("Bootstrap Sample 2 indices:", sample2_idx.tolist())

s1_x, s1_y = x[sample1_idx], y[sample1_idx]
s2_x, s2_y = x[sample2_idx], y[sample2_idx]

print("\nSample 1 (x, y):", list(zip(s1_x, s1_y)))
print("Sample 2 (x, y):", list(zip(s2_x, s2_y)))

In [ ]:
# 04 — Out-Of-Bag (OOB) indices
oob1 = [i for i in indices if i not in sample1_idx]
oob2 = [i for i in indices if i not in sample2_idx]

print("OOB indices for Sample 1:", oob1)
print("OOB indices for Sample 2:", oob2)

In [ ]:
# 05 — Mean of y for each bootstrap sample (base-learner predictions)
mean_s1 = s1_y.mean()
mean_s2 = s2_y.mean()

print(f"Mean of y — Sample 1: {mean_s1:.2f}")
print(f"Mean of y — Sample 2: {mean_s2:.2f}")

In [ ]:
# 06 — Aggregate (simple average) and compare with raw mean of all y
bagging_pred = (mean_s1 + mean_s2) / 2
raw_mean_y   = y.mean()

print(f"Bagging output (average of the two means): {bagging_pred:.2f}")
print(f"Raw mean of all y:                         {raw_mean_y:.2f}")
print(f"Difference: {abs(bagging_pred - raw_mean_y):.2f}")

### Q.1 Summary

| Item | Result |
|------|--------|
| Sample 1 indices | `[4, 0, 3, 3, 3]` |
| Sample 2 indices | `[1, 3, 2, 4, 0]` |
| OOB Sample 1 | `[1, 2]` |
| OOB Sample 2 | `[]` (empty – every index appeared at least once) |
| Mean y Sample 1 | 3.80 |
| Mean y Sample 2 | 4.00 |
| Bagging average | 3.90 |
| Raw mean of y | 4.00 |

## Q.2  Train & Aggregate Base Learners

In [ ]:
# Helper: fit the simple median-split rule on a bootstrap sample
def fit_median_rule(sx, sy):
    """Returns (median, left_mean, right_mean)"""
    med = np.median(sx)
    left_mask  = sx <= med
    right_mask = sx >  med
    left_mean  = sy[left_mask].mean()  if left_mask.any()  else np.nan
    right_mean = sy[right_mask].mean() if right_mask.any() else np.nan
    return med, left_mean, right_mean

def predict_median_rule(xx, med, left_mean, right_mean):
    return left_mean if xx <= med else right_mean

# 01–03 Fit both learners
med1, left1, right1 = fit_median_rule(s1_x, s1_y)
med2, left2, right2 = fit_median_rule(s2_x, s2_y)

print("Learner 1 (Sample 1)")
print(f"  median = {med1:.1f}")
print(f"  if x ≤ {med1:.1f} → predict {left1:.3f}")
print(f"  if x >  {med1:.1f} → predict {right1:.3f}")

print("\nLearner 2 (Sample 2)")
print(f"  median = {med2:.1f}")
print(f"  if x ≤ {med2:.1f} → predict {left2:.3f}")
print(f"  if x >  {med2:.1f} → predict {right2:.3f}")

In [ ]:
# 04 — New-point prediction for x = 25
x_new = 25.0
pred_l1 = predict_median_rule(x_new, med1, left1, right1)
pred_l2 = predict_median_rule(x_new, med2, left2, right2)

print(f"Learner 1 prediction for x=25: {pred_l1:.3f}")
print(f"Learner 2 prediction for x=25: {pred_l2:.3f}")

# 05 — Bagging output
bag_pred_25 = (pred_l1 + pred_l2) / 2
print(f"Bagging ensemble prediction for x=25: {bag_pred_25:.3f}")

In [ ]:
# 06 — SSE comparison on the original 5 points
preds_l1  = np.array([predict_median_rule(xi, med1, left1, right1) for xi in x])
preds_l2  = np.array([predict_median_rule(xi, med2, left2, right2) for xi in x])
preds_bag = (preds_l1 + preds_l2) / 2

sse_l1  = np.sum((y - preds_l1)**2)
sse_bag = np.sum((y - preds_bag)**2)

print("Predictions on original points:")
print(f"  Learner 1 : {np.round(preds_l1, 3)}")
print(f"  Learner 2 : {np.round(preds_l2, 3)}")
print(f"  Bagging   : {np.round(preds_bag, 3)}")
print()
print(f"SSE Learner 1 alone : {sse_l1:.3f}")
print(f"SSE Bagging ensemble: {sse_bag:.3f}")
print(f"\nBagging has the lower SSE → ensemble improves accuracy on the training data.")

### Q.2 Summary

| Item | Result |
|------|--------|
| Learner 1 rule | if x ≤ 40 → 3.500; else → 5.000 |
| Learner 2 rule | if x ≤ 30 → 3.667; else → 4.500 |
| Prediction at x=25 (L1) | 3.500 |
| Prediction at x=25 (L2) | 3.667 |
| Bagging prediction at x=25 | 3.583 |
| SSE Learner 1 | 5.000 |
| SSE Bagging | 4.750 |

## Q.3  Visualize It

In [ ]:
# Prepare continuous x-grid for the step-function lines
x_grid = np.linspace(5, 55, 500)

line_l1  = np.array([predict_median_rule(xi, med1, left1, right1) for xi in x_grid])
line_l2  = np.array([predict_median_rule(xi, med2, left2, right2) for xi in x_grid])
line_bag = (line_l1 + line_l2) / 2

# Markers: original points vs OOB of Sample 1
oob1_mask = np.isin(indices, oob1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- Scatter + step functions ----
ax = axes[0]
ax.scatter(x[~oob1_mask], y[~oob1_mask], c='steelblue', s=80, label='In Sample 1', zorder=5)
ax.scatter(x[oob1_mask],  y[oob1_mask],  c='orange',    s=80, marker='s',
           label='OOB Sample 1', zorder=5)

ax.plot(x_grid, line_l1,  color='green',  lw=2, label='Learner 1')
ax.plot(x_grid, line_l2,  color='purple', lw=2, label='Learner 2')
ax.plot(x_grid, line_bag, color='red',    lw=2.5, ls='--', label='Bagging ensemble')

ax.set_xlabel('x')
ax.set_ylabel('y / prediction')
ax.set_title('Scatter + Step-function Predictions')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_xlim(5, 55)

# ---- Bar chart of SSE ----
ax2 = axes[1]
bars = ax2.bar(['Learner 1', 'Bagging'], [sse_l1, sse_bag],
               color=['green', 'red'], alpha=0.8)
ax2.set_ylabel('SSE')
ax2.set_title('SSE Comparison (original 5 points)')
ax2.bar_label(bars, fmt='%.3f')
ax2.set_ylim(0, max(sse_l1, sse_bag)*1.25)
ax2.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Q.4  Explain It Cold

### 01  What is Bagging?

**Bagging** is an ensemble technique that builds many independent models on different random subsets of the training data and then combines their predictions (usually by averaging for regression or majority vote for classification) to produce a more stable final answer.

### 02  Purpose of sampling with replacement

Sampling **with replacement** (bootstrap) lets every observation have a chance to appear in every subset, so the subsets remain statistically representative of the full distribution.  
A simple non-overlapping split would force each subset to be smaller and would permanently exclude some points from certain models, increasing the variance of each individual learner and reducing the diversity that Bagging relies on.

### 04  How does Bagging reduce variance without significantly increasing bias?

Individual models trained on different bootstrap samples are highly **variable** (high variance) but have roughly the same **bias**.  
When we average many such models, the random fluctuations cancel out → variance shrinks (ideally by a factor of 1/M for M independent models).  
Because each base learner is still an unbiased (or equally biased) estimator of the same target function, the average inherits essentially the same bias.  
Hence Bagging moves us favorably along the bias-variance trade-off: lower variance, almost unchanged bias, lower total error.